<a href="https://colab.research.google.com/github/salmao555/lstm-lyrics-generator/blob/main/ltsm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RNN

In [3]:
import tensorflow as tf
import keras
import numpy as np

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("NumPy:", np.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.19.0
Keras: 3.13.2
NumPy: 2.0.2
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
from google.colab import files
uploaded = files.upload()


Saving JustinBieber.csv to JustinBieber.csv


In [5]:
import pandas as pd

df = pd.read_csv('JustinBieber.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head())

(348, 7)
['Unnamed: 0', 'Artist', 'Title', 'Album', 'Year', 'Date', 'Lyric']
   Unnamed: 0         Artist                   Title             Album  \
0           0  Justin Bieber           Love Yourself  Purpose (Deluxe)   
1           1  Justin Bieber                   Sorry  Purpose (Deluxe)   
2           2  Justin Bieber                   Yummy           Changes   
3           3  Justin Bieber  As Long as You Love Me           Believe   
4           4  Justin Bieber                    Baby      My World 2.0   

     Year        Date                                              Lyric  
0  2015.0  2015-11-13  produced by benny blanco   for all the times t...  
1  2015.0  2015-10-23  written by julia michaels justin tranter and j...  
2  2020.0  2020-01-03  yeah you got that yummyyum that yummyyum that ...  
3  2012.0  2012-06-11  justin bieber as long as you love me love me l...  
4  2010.0  2010-01-18  produced by thedream and tricky stewart   just...  


In [6]:
# grab only the Lyric column, drop any empty rows
lyrics = df['Lyric'].dropna().tolist()

print(f"Total songs: {len(lyrics)}")
print("\nSample lyric:")
print(lyrics[3])  # row 3 is "As Long as You Love Me" — cleaner lyrics

Total songs: 347

Sample lyric:
justin bieber as long as you love me love me love me love me love me love me love me as long as you love me love me love me love me love me as long as you love me   justin bieber we're under pressure we're under pressure seven billion people in the world tryna fit in tryna fit in keep it together keep it together smile on your face even though your heart is frowning frowning but hey now hey now you know girl you know girl we both know it's a cruel world cruel world but i will but i will take my chances   justin bieber as long as you love me we could be starving we could be homeless we could be broke as long as you love me i'll be your platinum i'll be your silver i'll be your gold as long as you love me love me as long as you love me love me   justin bieber i'll be your soldier i'll be your soldier fighting every second of the day for your dreams girl for your dreams girl i'll be your hova i'll be your hova you could be my destiny's child on the scene gi

In [7]:
# combine everything into one giant string
full_text = '\n'.join(lyrics)

print(f"Total characters: {len(full_text)}")
print("\nFirst 500 characters:")
print(full_text[:500])

Total characters: 532137

First 500 characters:
produced by benny blanco   for all the times that you rained on my parade and all the clubs you get in using my name you think you broke my heart oh girl for goodness sake you think i'm cryin' on my own well i ain't  refrain and i didn't wanna write a song 'cause i didn't want anyone thinking i still care i don't but you still hit my phone up and baby i'll be movin' on and i think you should be somethin' i don't wanna hold back maybe you should know that  pre my mama don't like you and she likes


In [8]:
import re

# lowercase everything
full_text = full_text.lower()

# keep only letters, spaces, newlines, basic punctuation
full_text = re.sub(r"[^a-z0-9\s\n',!?.]", '', full_text)

print(f"Characters after cleaning: {len(full_text)}")
print(full_text[:500])

Characters after cleaning: 532067
produced by benny blanco   for all the times that you rained on my parade and all the clubs you get in using my name you think you broke my heart oh girl for goodness sake you think i'm cryin' on my own well i ain't  refrain and i didn't wanna write a song 'cause i didn't want anyone thinking i still care i don't but you still hit my phone up and baby i'll be movin' on and i think you should be somethin' i don't wanna hold back maybe you should know that  pre my mama don't like you and she likes


In [9]:
# get all unique characters
chars = sorted(set(full_text))
vocab_size = len(chars)

print(f"Unique characters: {vocab_size}")
print(f"Characters: {''.join(chars)}")

# create two mappings
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print("\nExample mappings:")
print(f"'a' → {char_to_idx['a']}")
print(f"' ' → {char_to_idx[' ']}")

Unique characters: 39
Characters: 
 '0456789abcdefghijklmnopqrstuvwxyz   

Example mappings:
'a' → 10
' ' → 1


In [10]:
# turn every character in the text into its number
encoded = [char_to_idx[ch] for ch in full_text]

print(f"Total encoded characters: {len(encoded)}")
print(f"First 20 numbers: {encoded[:20]}")
print(f"Which spell out: {''.join([idx_to_char[i] for i in encoded[:20]])}")

Total encoded characters: 532067
First 20 numbers: [25, 27, 24, 13, 30, 12, 14, 13, 1, 11, 34, 1, 11, 14, 23, 23, 34, 1, 11, 21]
Which spell out: produced by benny bl


In [11]:
import numpy as np

seq_length = 40  # model looks at 40 characters to predict the next one

X = []
y = []

for i in range(len(encoded) - seq_length):
    X.append(encoded[i : i + seq_length])    # 40 chars input
    y.append(encoded[i + seq_length])         # 1 char to predict

print(f"Total sequences created: {len(X)}")
print(f"\nExample input:  {''.join([idx_to_char[i] for i in X[0]])}")
print(f"Example target: {idx_to_char[y[0]]}")

Total sequences created: 532027

Example input:  produced by benny blanco   for all the t
Example target: i


In [12]:
X = np.array(X)
y = np.array(y)

# normalize X to be between 0 and 1 (helps the model learn faster)
X = X / vocab_size

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (532027, 40)
y shape: (532027,)


In [13]:
# LSTM expects 3D input: (samples, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)
print(f"X reshaped: {X.shape}")

X reshaped: (532027, 40, 1)


In [14]:
import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(256, input_shape=(seq_length, 1), return_sequences=True),
    Dropout(0.2),        # randomly switches off 20% of neurons — prevents memorizing
    LSTM(128),           # second LSTM layer for deeper pattern learning
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')  # outputs probability for each character
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 40, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 39)             │         5,031 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 466,343 (1.78 MB)

 Trainable params: 466,343 (1.78 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam'
)

print("Model ready to train!")

Model ready to train!
